In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Bandingkan pengaturan transpiler

*Estimasi penggunaan: kurang dari satu menit pada prosesor Eagle r3 (CATATAN: Ini hanya estimasi. Waktu eksekusi aktual bisa berbeda-beda.)*

## Latar Belakang
Untuk memastikan hasil yang lebih cepat dan efisien, mulai 1 Maret 2024, Circuit dan observable perlu diubah agar hanya menggunakan instruksi yang didukung oleh QPU (quantum processing unit) sebelum dikirimkan ke primitif Qiskit Runtime. Kita menyebut ini sebagai Circuit dan observable *instruction set architecture* (ISA). Salah satu cara umum untuk melakukannya adalah menggunakan fungsi `generate_preset_pass_manager` dari Transpiler. Namun, kamu juga bisa memilih proses yang lebih manual.

Misalnya, kamu mungkin ingin menargetkan subset Qubit tertentu pada perangkat tertentu. Panduan ini menguji performa berbagai pengaturan Transpiler dengan menyelesaikan proses lengkap membuat, men-transpilasi, dan mengirimkan Circuit.

## Persyaratan
Sebelum memulai, pastikan kamu sudah menginstal hal-hal berikut:

* Qiskit SDK v1.2 atau lebih baru, dengan dukungan [visualisasi](https://docs.quantum.ibm.com/api/qiskit/visualization)
* Qiskit Runtime v0.28 atau lebih baru (`pip install qiskit-ibm-runtime`)

## Setup

In [ ]:
# Create circuit to test transpiler on
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.circuit.library import GroverOperator, Diagonal

# Use Statevector object to calculate the ideal output
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram
from qiskit.transpiler import PassManager

from qiskit.circuit.library import XGate
from qiskit.quantum_info import hellinger_fidelity

# Qiskit Runtime
from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    Batch,
    SamplerV2 as Sampler,
)
from qiskit_ibm_runtime.transpiler.passes.scheduling import (
    ASAPScheduleAnalysis,
    PadDynamicalDecoupling,
)

## Langkah 1: Petakan input klasik ke masalah kuantum
Buat Circuit kecil untuk dicoba dioptimalkan oleh Transpiler. Contoh ini membuat Circuit yang menjalankan algoritma Grover dengan oracle yang menandai state `111`. Selanjutnya, simulasikan distribusi ideal (apa yang kamu harapkan jika kamu menjalankan ini pada komputer kuantum sempurna sebanyak tak terhingga kali) untuk perbandingan nanti.

In [ ]:
# To run on hardware, select the backend with the fewest number of jobs in the queue
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
backend.name

'ibm_brisbanse'

In [29]:
oracle = Diagonal([1] * 7 + [-1])
qc = QuantumCircuit(3)
qc.h([0, 1, 2])
qc = qc.compose(GroverOperator(oracle))

qc.draw(output="mpl", style="iqp")

<Image src="../docs/images/guides/circuit-transpilation-settings/extracted-outputs/7e7944c5-68ac-40cf-a0eb-5f4a44d53931-0.avif" alt="Output of the previous code cell" />

In [30]:
ideal_distribution = Statevector.from_instruction(qc).probabilities_dict()

plot_histogram(ideal_distribution)

<Image src="../docs/images/guides/circuit-transpilation-settings/extracted-outputs/761afe09-b669-453f-8363-55070d6c8f57-0.avif" alt="Output of the previous code cell" />

## Step 2: Optimize problem for quantum hardware execution

Next, transpile the circuits for the QPU. You will compare the performance of the transpiler with `optimization_level` set to `0` (lowest) against `3` (highest). The lowest optimization level does the bare minimum needed to get the circuit running on the device; it maps the circuit qubits to the device qubits and adds swap gates to allow all two-qubit operations. The highest optimization level is much smarter and uses lots of tricks to reduce the overall gate count. Since multi-qubit gates have high error rates and qubits decohere over time, the shorter circuits should give better results.

The following cell transpiles `qc` for both values of `optimization_level`, prints the number of two-qubit gates, and adds the transpiled circuits to a list. Some of the transpiler's algorithms are randomized, so it sets a seed for reproducibility.

In [31]:
# Need to add measurements to the circuit
qc.measure_all()

# Find the correct two-qubit gate
twoQ_gates = set(["ecr", "cz", "cx"])
for gate in backend.basis_gates:
    if gate in twoQ_gates:
        twoQ_gate = gate

circuits = []
for optimization_level in [0, 3]:
    pm = generate_preset_pass_manager(
        optimization_level, backend=backend, seed_transpiler=0
    )
    t_qc = pm.run(qc)
    print(
        f"Two-qubit gates (optimization_level={optimization_level}): ",
        t_qc.count_ops()[twoQ_gate],
    )
    circuits.append(t_qc)

Two-qubit gates (optimization_level=0):  21
Two-qubit gates (optimization_level=3):  14


![Output of the previous code cell](../docs/images/guides/circuit-transpilation-settings/extracted-outputs/761afe09-b669-453f-8363-55070d6c8f57-0.avif)

## Langkah 2: Optimalkan masalah untuk eksekusi pada hardware kuantum
Selanjutnya, transpilasikan Circuit untuk QPU. Kamu akan membandingkan performa Transpiler dengan `optimization_level` yang diatur ke `0` (terendah) dan `3` (tertinggi). Level optimasi terendah hanya melakukan yang minimum yang diperlukan agar Circuit bisa berjalan di perangkat; ia memetakan Qubit Circuit ke Qubit perangkat dan menambahkan swap Gate untuk memungkinkan semua operasi dua Qubit. Level optimasi tertinggi jauh lebih cerdas dan menggunakan banyak trik untuk mengurangi jumlah Gate secara keseluruhan. Karena multi-Qubit Gate memiliki tingkat error yang tinggi dan Qubit mengalami dekoherensi seiring waktu, Circuit yang lebih pendek seharusnya memberikan hasil yang lebih baik.

Sel berikut men-transpilasikan `qc` untuk kedua nilai `optimization_level`, mencetak jumlah Gate dua Qubit, dan menambahkan Circuit yang telah di-transpilasi ke sebuah list. Beberapa algoritma Transpiler bersifat acak, sehingga seed ditetapkan untuk reprodusibilitas.

In [ ]:
# Get gate durations so the transpiler knows how long each operation takes
durations = backend.target.durations()

# This is the sequence we'll apply to idling qubits
dd_sequence = [XGate(), XGate()]

# Run scheduling and dynamic decoupling passes on circuit
pm = PassManager(
    [
        ASAPScheduleAnalysis(durations),
        PadDynamicalDecoupling(durations, dd_sequence),
    ]
)
circ_dd = pm.run(circuits[1])

# Add this new circuit to our list
circuits.append(circ_dd)

In [33]:
circ_dd.draw(output="mpl", style="iqp", idle_wires=False)

<Image src="../docs/images/guides/circuit-transpilation-settings/extracted-outputs/4ada6498-b9d7-4d88-b8a9-ef1dc0a85bf7-0.avif" alt="Output of the previous code cell" />

Karena CNOT biasanya memiliki tingkat error yang tinggi, Circuit yang di-transpilasi dengan `optimization_level=3` seharusnya berkinerja jauh lebih baik.

Cara lain untuk meningkatkan performa adalah melalui [dynamic decoupling](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.PadDynamicalDecoupling), dengan menerapkan serangkaian Gate pada Qubit yang sedang idle. Ini membatalkan beberapa interaksi yang tidak diinginkan dengan lingkungan. Sel berikut menambahkan dynamic decoupling ke Circuit yang di-transpilasi dengan `optimization_level=3` dan menambahkannya ke list.

In [34]:
with Batch(backend=backend):
    sampler = Sampler()
    job = sampler.run(
        [(circuit) for circuit in circuits],  # sample all three circuits
        shots=8000,
    )
    result = job.result()

## Step 4: Post-process and return result in desired classical format

Finally, plot the results from the device runs against the ideal distribution. You can see the results with `optimization_level=3` are closer to the ideal distribution due to the lower gate count, and `optimization_level=3 + dd` is even closer due to the dynamic decoupling.

In [35]:
binary_prob = [
    {
        k: v / res.data.meas.num_shots
        for k, v in res.data.meas.get_counts().items()
    }
    for res in result
]
plot_histogram(
    binary_prob + [ideal_distribution],
    bar_labels=False,
    legend=[
        "optimization_level=0",
        "optimization_level=3",
        "optimization_level=3 + dd",
        "ideal distribution",
    ],
)

<Image src="../docs/images/guides/circuit-transpilation-settings/extracted-outputs/525777ea-d438-4f3b-acb6-53e579f24a0e-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/circuit-transpilation-settings/extracted-outputs/4ada6498-b9d7-4d88-b8a9-ef1dc0a85bf7-0.avif)

## Langkah 3: Eksekusi menggunakan primitif Qiskit
Pada tahap ini, kamu punya list Circuit yang sudah di-transpilasi untuk QPU yang ditentukan. Selanjutnya, buat instance dari primitif Sampler dan mulai batched job menggunakan context manager (`with ...:`), yang secara otomatis membuka dan menutup batch.

Di dalam context manager, sample Circuit dan simpan hasilnya ke `result`.

In [ ]:
for prob in binary_prob:
    print(f"{hellinger_fidelity(prob, ideal_distribution):.3f}")

0.848
0.945
0.990
